In [ ]:
import pandas as pd
import openai
import re
import time
import os
from google.colab import userdata

# ==========================================
# 1. API & MODEL CONFIGURATION
# ==========================================
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = "sk-or-v1-YOUR-KEY-HERE"

client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# OpenRouter Nitro priority route for Qwen 2.5 72B
MODEL_NAME = "qwen/qwen-2.5-72b-instruct:nitro"

# ==========================================
# 2. SELECT DATASET (Uncomment ONE at a time)
# ==========================================
# INPUT_FILE, OUTPUT_FILE = "bangla_med_qa_correct.csv", "qwen_evaluated_bangla_med_qa_correct.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v1.csv", "qwen_evaluated_wrong_answers_v1.csv"
# INPUT_FILE, OUTPUT_FILE = "wrong_answers_v2.csv", "qwen_evaluated_wrong_answers_v2.csv"
INPUT_FILE, OUTPUT_FILE = "wrong_answers_v3.csv", "qwen_evaluated_wrong_answers_v3.csv"

# ==========================================
# 3. EVALUATION & PARSING FUNCTIONS
# ==========================================
def evaluate_pair(question, proposed_answer, retries=4):
    prompt = f"""You are a strict medical accuracy evaluator. Decide whether the provided model answer is correct for the question.
Only reply with a single digit: 1 or 0. No explanation, no punctuation, no extra text.
1 means the answer is factually correct and medically supported.
0 means the answer is hallucinated.

Question: {question}
Model answer: {proposed_answer}
Answer now:"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                # Progressive backoff delay when hitting API drops
                time.sleep(5 * (attempt + 1))
            else:
                return f"ERROR: {str(e)}"

def parse_binary_score(raw_text):
    if raw_text.startswith("ERROR:"):
        return 0
    # Strip any potential tags safely
    clean_text = re.sub(r'<.*?>', '', raw_text).strip()
    match = re.search(r'\b(0|1)\b', clean_text)
    if match:
        return int(match.group(1))
    return 1 if clean_text == "1" else 0

# ==========================================
# 4. EXECUTION LOOP
# ==========================================
def run_pipeline():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: File '{INPUT_FILE}' not found in current directory.")
        return

    print(f"Starting Evaluation on '{INPUT_FILE}' using model: {MODEL_NAME}")
    print("=" * 60)

    df = pd.read_csv(INPUT_FILE)

    raw_responses = []
    parsed_scores = []

    # Processing from Row 1 (Index 0)
    for idx, row in df.iterrows():
        question = row.get('question', '')
        answer = row.get('answer', '')

        raw_output = evaluate_pair(question, answer)
        score = parse_binary_score(raw_output)

        raw_responses.append(raw_output)
        parsed_scores.append(score)

        print(f"[{idx + 1}/{len(df)}] Score: {score} | Raw: '{raw_output}'")

        # 3-second delay buffer between rows to stay well clear of rate limits
        time.sleep(3.0)

    df['model_raw_response'] = raw_responses
    df['isCorrect'] = parsed_scores

    # Check if processing a wrong dataset or correct dataset
    is_wrong_dataset = "wrong" in INPUT_FILE.lower()

    if is_wrong_dataset:
        # Ground truth is 0 (Evaluator succeeds when isCorrect == 0)
        evaluator_correct_count = (df['isCorrect'] == 0).sum()
    else:
        # Ground truth is 1 (Evaluator succeeds when isCorrect == 1)
        evaluator_correct_count = (df['isCorrect'] == 1).sum()

    evaluator_accuracy = (evaluator_correct_count / len(df)) * 100

    df.to_csv(OUTPUT_FILE, index=False)
    print("=" * 60)
    print(f"✅ Processing Complete! Output saved to: '{OUTPUT_FILE}'")

    if is_wrong_dataset:
        print(f"   Evaluator Accuracy (Successfully flagged 0s): {evaluator_correct_count} / {len(df)} ({evaluator_accuracy:.2f}%)")
    else:
        print(f"   Evaluator Accuracy (Successfully flagged 1s): {evaluator_correct_count} / {len(df)} ({evaluator_accuracy:.2f}%)")

if __name__ == "__main__":
    run_pipeline()

In [ ]:
# ==========================================
# CELL 2: AUTO-RETRY LOOP FOR FAILED ROWS
# ==========================================

MAX_REPAIR_PASSES = 5  # Keeps trying until 0 errors or max passes reached
pass_num = 1

try:
    df = pd.read_csv(OUTPUT_FILE)
except FileNotFoundError:
    print(f"❌ Target CSV '{OUTPUT_FILE}' not found. Run Cell 1 first.")
    df = None

if df is not None:
    while pass_num <= MAX_REPAIR_PASSES:
        # 1. Check for remaining error rows
        error_mask = df['model_raw_response'].astype(str).str.contains("ERROR", case=False, na=True)
        error_indices = df[error_mask].index.tolist()

        if not error_indices:
            print(f"✨ Zero error rows remaining in '{OUTPUT_FILE}'! Dataset is 100% clean.")
            break

        print(f"\n🔄 [Pass {pass_num}/{MAX_REPAIR_PASSES}] Found {len(error_indices)} failed rows. Retrying...")
        print("=" * 60)

        for i, idx in enumerate(error_indices, start=1):
            question = df.loc[idx, 'question']
            answer = df.loc[idx, 'answer']

            # Evaluate with standard function
            raw_output = evaluate_pair(question, answer)
            score = parse_binary_score(raw_output)

            # Update dataframe in place
            df.loc[idx, 'model_raw_response'] = raw_output
            df.loc[idx, 'isCorrect'] = score

            print(f"[{i}/{len(error_indices)}] Row {idx + 1} -> Score: {score} | Raw: '{raw_output}'")

            # Checkpoint immediately to CSV
            df.to_csv(OUTPUT_FILE, index=False)
            time.sleep(3.0)

        pass_num += 1

    # 2. Dynamic Summary Calculation
    is_wrong_dataset = "wrong" in INPUT_FILE.lower()
    evaluator_correct_count = (df['isCorrect'] == 0 if is_wrong_dataset else df['isCorrect'] == 1).sum()
    evaluator_accuracy = (evaluator_correct_count / len(df)) * 100

    print("\n" + "=" * 60)
    print("📊 FINAL CLEAN DATASET METRICS:")
    print(f"   Dataset: {INPUT_FILE}")
    print(f"   Evaluator Accuracy: {evaluator_correct_count} / {len(df)} ({evaluator_accuracy:.2f}%)")